# Single-Environment MTPPO Experiment

This notebook is focused on one main comparison setting:

- one environment configuration;
- `20` retailers;
- configurable planning horizon in days;
- `MTPPO` versus a single heuristic baseline on the same setting.

The idea is to make iteration easier before launching larger benchmark sweeps.


In [5]:
from __future__ import annotations

import asyncio
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

if sys.platform.startswith('win'):
    try:
        asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())
    except AttributeError:
        pass

ROOT = Path.cwd().resolve().parents[0]
SRC = ROOT / 'src'
OUTPUTS = ROOT / 'outputs'
OUTPUTS.mkdir(exist_ok=True)

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from psrp_mtppo.baselines import evaluate_heuristic_baseline, run_heuristic_episode
from psrp_mtppo.config import EnvironmentConfig, ModelConfig, TrainingConfig
from psrp_mtppo.experiments import (
    plot_episode_dashboard,
    plot_retailer_heatmaps,
    plot_training_dashboard,
    run_joint_experiment,
)

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)


## Experiment Settings

Edit these values first. The notebook below uses them everywhere, so this is the main control panel for the experiment.


In [6]:
SHOW_PROGRESS = True
SEED = 42
TRACE_SEED = 123
DEVICE = 'cpu'

# Environment setup
NUM_RETAILERS = 20
HORIZON_DAYS = 7
HISTORY_LENGTH = 7
DEMAND_LOW = 1
DEMAND_HIGH = 50
RETAILER_CAPACITY = 100.0
INITIAL_RETAILER_INVENTORY = 100.0
SUPPLIER_INITIAL_INVENTORY = 1_000_000.0
VEHICLE_CAPACITY = 100.0
HOLDING_COST = 2.0
PRODUCT_PRICE = 20.0
SALES_LOSS_PENALTY = 0.3
TRANSPORT_COST_PER_DISTANCE = 1000.0

# Model setup
GIN_DIMS = (64, 128, 128)
MLP_DIMS = (128, 128)
STATE_EMBED_DIM = 128
ATTENTION_HEADS = 8
ATTENTION_LAYERS = 2
DROPOUT = 0.1

# Training setup
LEARNING_RATE = 1e-3
GAMMA = 0.9
TRAIN_BATCH_SIZE = 256
MINIBATCH_SIZE = 64
PPO_EPOCHS = 4
TRAINING_ITERATIONS = 10
EVAL_EPISODES = 3
CLIP_PARAM = 0.2
VALUE_CLIP_PARAM = 0.1
KL_COEFFICIENT = 0.2
ENTROPY_COEFFICIENT = 0.01
MAX_GRAD_NORM = 1.0

# Baseline setup
BASELINE_SOLVER = 'ortools'  # 'ortools' or 'greedy'


In [7]:
ENV_CONFIG = EnvironmentConfig(
    num_retailers=NUM_RETAILERS,
    horizon_days=HORIZON_DAYS,
    history_length=HISTORY_LENGTH,
    holding_cost=HOLDING_COST,
    retailer_capacity=RETAILER_CAPACITY,
    initial_retailer_inventory=INITIAL_RETAILER_INVENTORY,
    supplier_initial_inventory=SUPPLIER_INITIAL_INVENTORY,
    demand_low=DEMAND_LOW,
    demand_high=DEMAND_HIGH,
    product_price=PRODUCT_PRICE,
    sales_loss_penalty=SALES_LOSS_PENALTY,
    vehicle_capacity=VEHICLE_CAPACITY,
    transport_cost_per_distance=TRANSPORT_COST_PER_DISTANCE,
    seed=SEED,
)

MODEL_CONFIG = ModelConfig(
    gin_dims=GIN_DIMS,
    mlp_dims=MLP_DIMS,
    state_embed_dim=STATE_EMBED_DIM,
    attention_heads=ATTENTION_HEADS,
    attention_layers=ATTENTION_LAYERS,
    dropout=DROPOUT,
)

TRAIN_CONFIG = TrainingConfig(
    learning_rate=LEARNING_RATE,
    gamma=GAMMA,
    train_batch_size=TRAIN_BATCH_SIZE,
    minibatch_size=MINIBATCH_SIZE,
    ppo_epochs=PPO_EPOCHS,
    training_iterations=TRAINING_ITERATIONS,
    evaluation_episodes=EVAL_EPISODES,
    clip_param=CLIP_PARAM,
    value_clip_param=VALUE_CLIP_PARAM,
    kl_coefficient=KL_COEFFICIENT,
    entropy_coefficient=ENTROPY_COEFFICIENT,
    max_grad_norm=MAX_GRAD_NORM,
    device=DEVICE,
    seed=SEED,
)

config_table = pd.DataFrame(
    {
        'group': ['env'] * 6 + ['model'] * 5 + ['train'] * 9,
        'name': [
            'num_retailers', 'horizon_days', 'demand_low', 'demand_high', 'vehicle_capacity', 'transport_cost_per_distance',
            'gin_dims', 'mlp_dims', 'state_embed_dim', 'attention_heads', 'dropout',
            'learning_rate', 'gamma', 'train_batch_size', 'minibatch_size', 'ppo_epochs', 'training_iterations', 'evaluation_episodes', 'device', 'seed',
        ],
        'value': [
            NUM_RETAILERS, HORIZON_DAYS, DEMAND_LOW, DEMAND_HIGH, VEHICLE_CAPACITY, TRANSPORT_COST_PER_DISTANCE,
            GIN_DIMS, MLP_DIMS, STATE_EMBED_DIM, ATTENTION_HEADS, DROPOUT,
            LEARNING_RATE, GAMMA, TRAIN_BATCH_SIZE, MINIBATCH_SIZE, PPO_EPOCHS, TRAINING_ITERATIONS, EVAL_EPISODES, DEVICE, SEED,
        ],
    }
)
config_table


,group,name,value
0,env,num_retailers,20
1,env,horizon_days,7
2,env,demand_low,1
3,env,demand_high,50
4,env,vehicle_capacity,100.0
5,env,transport_cost_per_distance,1000.0
6,model,gin_dims,"(64, 128, 128)"
7,model,mlp_dims,"(128, 128)"
8,model,state_embed_dim,128
9,model,attention_heads,8


## Train MTPPO on the single 20-retailer setting

This is the main training run for the notebook.


In [ ]:
trainer, history = run_joint_experiment(
    env_config=ENV_CONFIG,
    model_config=MODEL_CONFIG,
    training_config=TRAIN_CONFIG,
    show_progress=SHOW_PROGRESS,
)
history.to_csv(OUTPUTS / 'single_env_history.csv', index=False)
history


D:\PythonProjects\PSRP_D3PO\src\psrp_mtppo\models\attention.py:10: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(
MTPPO train (20 retailers):  20%|██        | 2/10 [02:10<08:47, 65.92s/it, eval_cost=132758.79, fill_rate=1.000, mean_return=-51731.58]

In [ ]:
history[['iteration', 'mean_return', 'eval_sum_cost', 'eval_fill_rate', 'inventory_actor_loss', 'routing_actor_loss', 'critic_loss']].round(3)


In [ ]:
plot_training_dashboard(history)
plt.show()


## Compare MTPPO against one baseline on the same environment

Here we compare on the same `20`-retailer configuration, not across multiple scales.


In [ ]:
mtppo_episode_rows = []
for eval_seed in range(100, 100 + EVAL_EPISODES):
    episode_summary, _episode_trace = trainer.rollout_episode(seed=eval_seed, greedy=True, show_progress=False)
    mtppo_episode_rows.append(episode_summary)

mtppo_eval_df = pd.DataFrame(mtppo_episode_rows)
baseline_eval_df = evaluate_heuristic_baseline(
    ENV_CONFIG,
    episodes=EVAL_EPISODES,
    route_solver=BASELINE_SOLVER,
)

comparison_mean = pd.DataFrame(
    [
        {'algorithm': 'MTPPO', **mtppo_eval_df.mean(numeric_only=True).to_dict()},
        {'algorithm': f'Heuristic-{BASELINE_SOLVER}', **baseline_eval_df.mean(numeric_only=True).to_dict()},
    ]
).set_index('algorithm')

comparison_std = pd.DataFrame(
    [
        {'algorithm': 'MTPPO', **mtppo_eval_df.std(numeric_only=True).fillna(0.0).to_dict()},
        {'algorithm': f'Heuristic-{BASELINE_SOLVER}', **baseline_eval_df.std(numeric_only=True).fillna(0.0).to_dict()},
    ]
).set_index('algorithm')

comparison_mean.round(3)


In [ ]:
comparison_std.round(3)


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for axis, metric in zip(axes, ['reward', 'sum_cost', 'route_distance', 'fill_rate']):
    comparison_mean[metric].plot(kind='bar', ax=axis, rot=0, title=metric.replace('_', ' ').title())
    axis.grid(alpha=0.3)
fig.tight_layout()
plt.show()


## Detailed episode trace on a fixed seed

This section is useful for debugging behavior day by day on the exact same environment instance.


In [ ]:
mtppo_trace_summary, mtppo_trace = trainer.rollout_episode(
    seed=TRACE_SEED,
    greedy=True,
    show_progress=SHOW_PROGRESS,
)

baseline_trace_summary, baseline_trace = run_heuristic_episode(
    ENV_CONFIG,
    seed=TRACE_SEED,
    route_solver=BASELINE_SOLVER,
)

trace_summary = pd.DataFrame(
    [
        {'algorithm': 'MTPPO', **mtppo_trace_summary},
        {'algorithm': f'Heuristic-{BASELINE_SOLVER}', **baseline_trace_summary},
    ]
).set_index('algorithm')
trace_summary.round(3)


In [ ]:
plot_episode_dashboard(mtppo_trace['daily'], title_prefix='MTPPO')
plt.show()


In [ ]:
plot_episode_dashboard(baseline_trace['daily'], title_prefix=f'Heuristic-{BASELINE_SOLVER}')
plt.show()


In [ ]:
mtppo_daily = mtppo_trace['daily'].copy().rename(columns=lambda column: f'mtppo_{column}' if column != 'day' else column)
baseline_daily = baseline_trace['daily'].copy().rename(columns=lambda column: f'baseline_{column}' if column != 'day' else column)
daily_compare = mtppo_daily.merge(baseline_daily, on='day', how='inner')

fig, axes = plt.subplots(2, 2, figsize=(16, 9))
axes[0, 0].plot(daily_compare['day'], daily_compare['mtppo_reward'], label='MTPPO')
axes[0, 0].plot(daily_compare['day'], daily_compare['baseline_reward'], label=f'Heuristic-{BASELINE_SOLVER}')
axes[0, 0].set_title('Daily Reward')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(daily_compare['day'], daily_compare['mtppo_route_cost'], label='MTPPO')
axes[0, 1].plot(daily_compare['day'], daily_compare['baseline_route_cost'], label=f'Heuristic-{BASELINE_SOLVER}')
axes[0, 1].set_title('Daily Route Cost')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

axes[1, 0].plot(daily_compare['day'], daily_compare['mtppo_fill_rate'], label='MTPPO')
axes[1, 0].plot(daily_compare['day'], daily_compare['baseline_fill_rate'], label=f'Heuristic-{BASELINE_SOLVER}')
axes[1, 0].set_title('Daily Fill Rate')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(daily_compare['day'], daily_compare['mtppo_total_replenishment'], label='MTPPO')
axes[1, 1].plot(daily_compare['day'], daily_compare['baseline_total_replenishment'], label=f'Heuristic-{BASELINE_SOLVER}')
axes[1, 1].set_title('Daily Total Replenishment')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

fig.tight_layout()
plt.show()


## Retailer-level heatmaps

These plots help inspect which retailers are being replenished, how demand evolves, and where inventory remains high or low over time.


In [ ]:
plot_retailer_heatmaps(
    ending_inventory=mtppo_trace['ending_inventory'],
    replenishment=mtppo_trace['replenishment'],
    demand=mtppo_trace['demand'],
    title_prefix='MTPPO',
)
plt.show()


In [ ]:
plot_retailer_heatmaps(
    ending_inventory=baseline_trace['ending_inventory'],
    replenishment=baseline_trace['replenishment'],
    demand=baseline_trace['demand'],
    title_prefix=f'Heuristic-{BASELINE_SOLVER}',
)
plt.show()


In [ ]:
comparison_mean.to_csv(OUTPUTS / 'single_env_comparison_mean.csv')
comparison_std.to_csv(OUTPUTS / 'single_env_comparison_std.csv')
mtppo_trace['daily'].to_csv(OUTPUTS / 'single_env_mtppo_daily_trace.csv', index=False)
baseline_trace['daily'].to_csv(OUTPUTS / 'single_env_baseline_daily_trace.csv', index=False)
print('Saved notebook artifacts to', OUTPUTS)
